# 🤖 Telecom Customer Churn — Modeling
## Notebook 2 of 2 — Feature Engineering → Modeling → Evaluation → Business Insights

---

| Item | Detail |
|------|--------|
| **Project** | Customer Churn Prediction — Telecom |
| **Input** | `../data/processed/telecom_cleaned.csv` (from Notebook 1) |
| **Algorithm** | Logistic Regression (Statsmodels GLM + Scikit-learn) |
| **Model Output** | `../models/churn_model.pkl` |
| **Previous Notebook** | `01_EDA_ML.ipynb` |

---

# --- Smart Path Resolution (works in VS Code, Jupyter, and Colab) ---
import os
from pathlib import Path

if os.path.exists('../data/processed'):
    DATA_PROCESSED = Path('../data/processed')
    MODELS_DIR = Path('../models')
elif os.path.exists('data/processed'):
    DATA_PROCESSED = Path('data/processed')
    MODELS_DIR = Path('models')
else:
    DATA_PROCESSED = Path('.')
    MODELS_DIR = Path('.')

## 📋 Notebook Structure

```
1.  Import Libraries
2.  Load Cleaned Data (from Notebook 1)
3.  Feature Engineering
    3a. Binary encoding (Yes/No → 1/0)
    3b. Target isolation
    3c. Churn baseline rate
4.  Train-Test Split Strategy
5.  One-Hot Encoding (Categorical → Dummies)
6.  Multicollinearity Removal (Redundant dummy drop)
7.  Feature Scaling (StandardScaler)
8.  Post-Scaling Correlation Heatmap
9.  Feature Selection — Recursive Feature Elimination (RFE)
10. Model Building — Iterative GLM with VIF/p-value stopping criteria
    10a. Model 1
    10b. Model 2 (drop MonthlyCharges)
    10c. Model 3 — Final (drop TotalCharges) 
11. Training Set Predictions & Evaluation
12. ROC Curve & AUC
13. Optimal Threshold Selection
    13a. Precision-Recall Curve (PR Cutoff)
    13b. Sensitivity / Specificity / Accuracy Grid
14. Test Set Evaluation (Generalization Check)
15. Model Stability Summary (Train vs. Test)
16. Model Saving (Joblib + Pickle)
17. Model Inferencing (New Customer Prediction)
18. Business Insights & Recommendations
```

---
## 1. 📦 Import Libraries

In [ ]:
from IPython.display import display
%matplotlib inline
# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Statsmodels (GLM for statistical inference)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn import metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve
)

# Model persistence
import joblib
import pickle
import os

print("✅ All libraries loaded successfully.")

---
## 2. 📂 Load Cleaned Dataset (from Notebook 1)

In [ ]:
# --- Smart Path Resolution (works in VS Code, Jupyter, and Colab) ---
import os
from pathlib import Path

if os.path.exists('../data/processed'):
    DATA_PROCESSED = Path('../data/processed')
    MODELS_DIR = Path('../models')
elif os.path.exists('data/processed'):
    DATA_PROCESSED = Path('data/processed')
    MODELS_DIR = Path('models')
else:
    DATA_PROCESSED = Path('.')
    MODELS_DIR = Path('.')

telecom = pd.read_csv(DATA_PROCESSED / 'telecom_cleaned.csv')
print(f"Loaded dataset: {telecom.shape[0]:,} rows × {telecom.shape[1]} columns")
telecom.head(3)


---
## 3. 🔧 Feature Engineering

### 3a. Binary Encoding (Yes/No → 1/0)

Several features are binary Yes/No strings. We map them to 1/0 integers for model compatibility.

In [ ]:
# Columns with simple Yes/No encoding
binary_cols = ['PhoneService', 'PaperlessBilling', 'Churn', 'Partner', 'Dependents']

telecom[binary_cols] = telecom[binary_cols].apply(
    lambda col: col.map({'Yes': 1, 'No': 0})
)

print("Binary encoding applied:")
telecom[binary_cols].head(3)

### 3b. Target Isolation & Feature Matrix

In [ ]:
# Separate features and target — drop customerID (identifier, not a predictor)
X = telecom.drop(columns=['Churn', 'customerID'])
y = telecom['Churn']

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}")

### 3c. Baseline Churn Rate

In [ ]:
baseline_churn_rate = (y.sum() / len(y)) * 100
print(f"📊 Baseline Churn Rate : {baseline_churn_rate:.2f}%")
print(f"   Churned customers   : {y.sum():,} / {len(y):,}")

---
## 4. ✂️ Train-Test Split Strategy

**Strategy chosen**: Stratified 70/30 random split with `random_state=100`

| Split | Rows | Churn %  | Rationale |
|-------|------|----------|-----------|
| **Train** (70%) | 4,922 | ~26.6% | Enough data for stable GLM coefficients and RFE |
| **Test** (30%) | 2,110 | ~26.6% | Large enough for reliable generalization metrics |

> **Why not cross-validation?** The dataset is sufficiently large (7,032 records) and logistic regression is a low-variance model. A single holdout provides clean, interpretable results suited for the business presentation context.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.70, test_size=0.30, random_state=100
)

# Ensure we work on independent copies to prevent slice assignment warnings
X_train = X_train.copy()
X_test  = X_test.copy()

print(f"Train set : {X_train.shape[0]:,} rows  |  Churn rate: {y_train.mean()*100:.2f}%")
print(f"Test set  : {X_test.shape[0]:,} rows  |  Churn rate: {y_test.mean()*100:.2f}%")

---
## 5. 🔡 One-Hot Encoding (Categorical → Dummies)

We use `pd.get_dummies` with `drop_first=True` to:
1. Eliminate the **dummy variable trap** (perfect multicollinearity)
2. Apply independently on train/test to prevent **data leakage**
3. Use `.reindex()` to force test to match train's feature blueprint

In [ ]:
# Identify all object/categorical columns automatically
cat_cols = list(X_train.select_dtypes(include=['object']).columns)
print(f"Categorical columns to encode ({len(cat_cols)}):")
for c in cat_cols:
    print(f"  → {c}: {X_train[c].unique().tolist()}")

In [ ]:
# Encode training set
X_train = pd.get_dummies(data=X_train, columns=cat_cols, drop_first=True, dtype=int)

# Encode test set
X_test = pd.get_dummies(data=X_test, columns=cat_cols, drop_first=True, dtype=int)

# Align test set structure to train set — fill missing dummies with 0
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"X_train after encoding : {X_train.shape}")
print(f"X_test  after encoding : {X_test.shape}")
print(f"\nAll features:")
print(list(X_train.columns))

---
## 6. 🧹 Multicollinearity Removal — Redundant Dummy Drop

Several dummy-encoded categories are **logically redundant** because they are perfectly co-determined by another feature:

| Column to Drop | Why |
|----------------|----|
| `MultipleLines_No phone service` | = When `PhoneService = 0` |
| `OnlineSecurity_No internet service` | = When `InternetService_No = 1` |
| `OnlineBackup_No internet service` | Same as above |
| `DeviceProtection_No internet service` | Same as above |
| `TechSupport_No internet service` | Same as above |
| `StreamingTV_No internet service` | Same as above |
| `StreamingMovies_No internet service` | Same as above |

In [ ]:
cols_to_drop = [
    'MultipleLines_No phone service',
    'OnlineSecurity_No internet service',
    'OnlineBackup_No internet service',
    'DeviceProtection_No internet service',
    'TechSupport_No internet service',
    'StreamingTV_No internet service',
    'StreamingMovies_No internet service'
]

# errors='ignore' prevents crash if any column is already absent
X_train.drop(columns=cols_to_drop, inplace=True, errors='ignore')
X_test.drop(columns=cols_to_drop,  inplace=True, errors='ignore')

print(f"X_train after redundancy drop : {X_train.shape}")
print(f"X_test  after redundancy drop : {X_test.shape}")

---
## 7. 📏 Feature Scaling (StandardScaler)

Logistic regression with regularization is sensitive to feature scales. We standardize continuous features.

> **Important**: Fit the scaler **only on train data**, then apply the same transform to test data to prevent data leakage.

In [ ]:
scaler   = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Fit on train ONLY, transform both
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])  # use train's μ, σ

print("Scaled feature statistics (train):")
X_train[num_cols].describe().round(4)

---
## 8. 🌡️ Post-Encoding Correlation Heatmap

In [ ]:
plt.figure(figsize=(18, 10))
corr_matrix = X_train.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, cmap='coolwarm', fmt='.2f', annot=True,
            mask=mask, linewidths=0.3, annot_kws={'size': 7},
            cbar_kws={'shrink': 0.7})
plt.title('Post-Encoding Training Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Observation**: `MonthlyCharges` and `TotalCharges` show high correlation — we expect VIF analysis to flag this, and we will drop accordingly during the iterative model building phase.

---
## 9. 🎯 Feature Selection — Recursive Feature Elimination (RFE)

RFE wraps a base Logistic Regression estimator and iteratively removes the weakest features (by coefficient magnitude) until `n_features_to_select` remain.

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
rfe    = RFE(estimator=logreg, n_features_to_select=15)
rfe.fit(X_train, y_train)

# Display RFE results
rfe_results = pd.DataFrame({
    'Feature'  : X_train.columns,
    'Selected' : rfe.support_,
    'Rank'     : rfe.ranking_
}).sort_values('Rank')

print("\n📌 RFE Feature Selection Results:")
display(rfe_results)

In [ ]:
# Extract the 15 selected features
rfe_cols = X_train.columns[rfe.support_]
print(f"Selected {len(rfe_cols)} features:")
print(list(rfe_cols))

---
## 10. 🏗️ Model Building — Iterative GLM (Stopping Criteria: p-value & VIF)

**Stopping Criteria:**  
✅ All features have **p-value < 0.05** (statistically significant)  
✅ All features have **VIF < 5** (no problematic multicollinearity)

We use `statsmodels.GLM` with Binomial family (Logit link) for detailed statistical summaries.

In [ ]:
def compute_vif(X_subset):
    """Returns a DataFrame of VIF scores for each feature."""
    vif = pd.DataFrame()
    vif['Feature'] = X_subset.columns
    vif['VIF'] = [
        variance_inflation_factor(X_subset.values, i)
        for i in range(X_subset.shape[1])
    ]
    vif['VIF'] = vif['VIF'].round(2)
    return vif.sort_values('VIF', ascending=False).reset_index(drop=True)

### 10a. Model 1 — All 15 RFE Features

In [ ]:
X_train_sm = sm.add_constant(X_train[rfe_cols])
logm1 = sm.GLM(y_train, X_train_sm, family=sm.families.Binomial())
model1 = logm1.fit()
print(model1.summary())

In [ ]:
print("\nVIF Analysis — Model 1:")
display(compute_vif(X_train[rfe_cols]))

> **Action**: `MonthlyCharges` has the highest VIF (> 10). Drop it → Model 2.

### 10b. Model 2 — Drop MonthlyCharges

In [ ]:
rfe_cols = rfe_cols.drop('MonthlyCharges')
print(f"Features in Model 2 ({len(rfe_cols)}): {list(rfe_cols)}")

In [ ]:
X_train_sm1 = sm.add_constant(X_train[rfe_cols])
logm2 = sm.GLM(y_train, X_train_sm1, family=sm.families.Binomial())
model2 = logm2.fit()
print(model2.summary())

In [ ]:
print("\nVIF Analysis — Model 2:")
display(compute_vif(X_train[rfe_cols]))

> **Action**: `TotalCharges` still has VIF > 5. Drop it → Model 3.

### 10c. Model 3 — Final Model (Drop TotalCharges)

In [ ]:
rfe_cols = rfe_cols.drop('TotalCharges')
print(f"Final features ({len(rfe_cols)}): {list(rfe_cols)}")

In [ ]:
X_train_sm2 = sm.add_constant(X_train[rfe_cols])
logm3 = sm.GLM(y_train, X_train_sm2, family=sm.families.Binomial())
model3 = logm3.fit()
print(model3.summary())

In [ ]:
print("\nVIF Analysis — Model 3 (Final):")
vif_final = compute_vif(X_train[rfe_cols])
display(vif_final)

print(f"\n✅ Stopping Criteria Met:")
print(f"   Max VIF = {vif_final['VIF'].max():.2f} (threshold: 5.0)")
print(f"   All p-values < 0.05 (see model summary above)")

---
## 11. 📊 Training Set Predictions & Initial Evaluation

### 11a. Predict Probabilities on Training Set

In [ ]:
# Predict probabilities on training set
y_train_pred = model3.predict(X_train_sm2)

# Create results DataFrame — y_train_pred is a numpy array, no .values needed
train_results = pd.DataFrame({
    'Actual'      : y_train.values,
    'Probability' : y_train_pred
})

# Capture train AUC here for later use in model_package
train_auc = roc_auc_score(train_results['Actual'], train_results['Probability'])

print(f"Training predictions shape: {train_results.shape}")
print(f"Training AUC: {train_auc:.4f}")
train_results.head(10)


---
## 12. 📈 ROC Curve & AUC Score

In [ ]:
def plot_roc_curve(actual, probs, title='ROC Curve', ax=None):
    """Plots ROC curve with AUC score."""
    fpr, tpr, _ = roc_curve(actual, probs)
    auc_score   = roc_auc_score(actual, probs)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))
    
    ax.plot(fpr, tpr, color='#1565C0', linewidth=2.5,
            label=f'ROC Curve  (AUC = {auc_score:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.50)')
    ax.fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate (1 − Specificity)', fontsize=12)
    ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(True, alpha=0.3)
    return auc_score

# Plot Training ROC curve
plot_roc_curve(train_results['Actual'], train_results['Probability'],
               title='ROC Curve — Training Set')
plt.tight_layout()
plt.show()
print(f"\n📊 Training AUC Score: {train_auc:.4f}")


> **Interpretation**: An AUC of ~0.85 means: if we randomly pick one churned customer and one loyal customer, the model correctly assigns a **higher churn risk score to the true churner 85% of the time**. Industry benchmark: 0.80–0.90 = highly stable and production-ready.

---
## 13. 🎚️ Optimal Threshold Selection

The default threshold of 0.50 is not optimal for imbalanced business problems. We evaluate two methods:

### 13a. Precision-Recall Curve (PR Curve)

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(
    train_results['Actual'], train_results['Probability']
)

plt.figure(figsize=(10, 6))
plt.plot(pr_thresholds, precision[:-1], 'g-', linewidth=2, label='Precision')
plt.plot(pr_thresholds, recall[:-1],    'r-', linewidth=2, label='Recall (Sensitivity)')

# Find PR crossover
diff = np.abs(precision[:-1] - recall[:-1])
pr_optimal_idx = np.argmin(diff)
pr_optimal_thresh = pr_thresholds[pr_optimal_idx]
plt.axvline(x=pr_optimal_thresh, color='purple', linestyle='--',
            label=f'PR Optimal Threshold = {pr_optimal_thresh:.2f}')

plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision-Recall Trade-off vs. Threshold', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"PR Curve Optimal Threshold: {pr_optimal_thresh:.2f}")
print(f"  Precision at {pr_optimal_thresh:.2f}: {precision[pr_optimal_idx]:.3f}")
print(f"  Recall    at {pr_optimal_thresh:.2f}: {recall[pr_optimal_idx]:.3f}")

### 13b. Sensitivity / Specificity / Accuracy Grid

We evaluate all metrics across a range of thresholds to find the **optimal operational balance**.

In [ ]:
# Create threshold evaluation grid
thresholds = np.arange(0.0, 1.0, 0.05)
cutoff_rows = []

for t in thresholds:
    preds = (train_results['Probability'] > t).astype(int)
    cm    = confusion_matrix(train_results['Actual'], preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        total = tn + fp + fn + tp
        accuracy    = (tp + tn) / total
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        cutoff_rows.append({
            'Threshold'  : round(t, 2),
            'Accuracy'   : round(accuracy, 4),
            'Sensitivity': round(sensitivity, 4),
            'Specificity': round(specificity, 4)
        })

cutoff_df = pd.DataFrame(cutoff_rows)
print("Threshold Evaluation Grid:")
display(cutoff_df)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(cutoff_df['Threshold'], cutoff_df['Accuracy'],    'b-o', markersize=5, label='Accuracy')
plt.plot(cutoff_df['Threshold'], cutoff_df['Sensitivity'], 'r-o', markersize=5, label='Sensitivity (Recall)')
plt.plot(cutoff_df['Threshold'], cutoff_df['Specificity'], 'g-o', markersize=5, label='Specificity')

# Find where Sensitivity ≈ Specificity (intersection)
diff_ss = np.abs(cutoff_df['Sensitivity'] - cutoff_df['Specificity'])
optimal_idx    = diff_ss.idxmin()
OPTIMAL_THRESH = cutoff_df.loc[optimal_idx, 'Threshold']

plt.axvline(x=OPTIMAL_THRESH, color='purple', linestyle='--', linewidth=2,
            label=f'Optimal Threshold = {OPTIMAL_THRESH:.2f}')
plt.scatter([OPTIMAL_THRESH],
            [cutoff_df.loc[optimal_idx, 'Sensitivity']], s=120, color='purple', zorder=5)

plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('Metric Score', fontsize=12)
plt.title('Accuracy / Sensitivity / Specificity vs. Threshold', fontsize=13, fontweight='bold')
plt.legend(loc='center right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

row = cutoff_df.loc[optimal_idx]
print(f"\n🎯 Optimal Threshold Selected: {OPTIMAL_THRESH}")
print(f"   Sensitivity (Recall)  : {row['Sensitivity']:.4f}")
print(f"   Specificity           : {row['Specificity']:.4f}")
print(f"   Accuracy              : {row['Accuracy']:.4f}")

### 13c. Apply Optimal Threshold to Training Set

In [ ]:
train_results['Predicted'] = (train_results['Probability'] > OPTIMAL_THRESH).astype(int)

# Training Classification Report
print(f"=== Training Set Classification Report (Threshold = {OPTIMAL_THRESH}) ===")
print(classification_report(train_results['Actual'], train_results['Predicted'],
                             target_names=['Not Churn (0)', 'Churn (1)']))

# Confusion Matrix
cm_train = confusion_matrix(train_results['Actual'], train_results['Predicted'])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: No Churn', 'Pred: Churn'],
            yticklabels=['True: No Churn', 'True: Churn'])
ax.set_title(f'Confusion Matrix — Training Set (Threshold={OPTIMAL_THRESH})', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 14. 🧪 Test Set Evaluation (Generalization Check)

In [ ]:
# Subset X_test to the final selected features
X_test_final = X_test[rfe_cols]
X_test_sm    = sm.add_constant(X_test_final, has_constant='add')

# Predict probabilities on test set (returns numpy array)
y_test_prob  = model3.predict(X_test_sm)

# Apply optimal threshold
y_test_pred  = (y_test_prob > OPTIMAL_THRESH).astype(int)

# Build results DataFrame (numpy arrays, no .values needed)
test_results = pd.DataFrame({
    'Actual'     : y_test.values,
    'Probability': y_test_prob,
    'Predicted'  : y_test_pred
})

# Capture test AUC for model_package
test_auc = roc_auc_score(test_results['Actual'], test_results['Probability'])

print(f"=== Test Set Classification Report (Threshold = {OPTIMAL_THRESH}) ===")
print(classification_report(test_results['Actual'], test_results['Predicted'],
                             target_names=['Not Churn (0)', 'Churn (1)']))
print(f"Test Set AUC: {test_auc:.4f}")


In [ ]:
# Test Set ROC + Confusion Matrix side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curve
plot_roc_curve(test_results['Actual'], test_results['Probability'],
               title='ROC Curve — Test Set', ax=axes[0])

# Confusion matrix
cm_test = confusion_matrix(test_results['Actual'], test_results['Predicted'])
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Pred: No Churn', 'Pred: Churn'],
            yticklabels=['True: No Churn', 'True: Churn'])
axes[1].set_title(f'Confusion Matrix — Test Set (Threshold={OPTIMAL_THRESH})', fontweight='bold')

plt.tight_layout()
plt.show()
print(f"\nTest Set AUC: {test_auc:.4f}")


---
## 15. 📊 Model Stability Summary (Train vs. Test)

In [ ]:
# Detailed metric comparison
train_cm = confusion_matrix(train_results['Actual'], train_results['Predicted'])
test_cm  = confusion_matrix(test_results['Actual'],  test_results['Predicted'])

def metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    return {
        'Accuracy'           : round((tp + tn) / (tn + fp + fn + tp), 4),
        'Sensitivity/Recall' : round(tp / (tp + fn), 4),
        'Specificity'        : round(tn / (tn + fp), 4),
        'Precision'          : round(tp / (tp + fp), 4),
        'F1-Score (Churn)'   : round(2 * tp / (2 * tp + fp + fn), 4)
    }

summary = pd.DataFrame({
    'Train Set': metrics_from_cm(train_cm),
    'Test Set' : metrics_from_cm(test_cm)
})

print(f"=== Model Stability Summary (Threshold = {OPTIMAL_THRESH}) ===")
display(summary)

accuracy_drop = summary.loc['Accuracy', 'Train Set'] - summary.loc['Accuracy', 'Test Set']
print(f"\n✅ VERDICT: The model generalizes well.")
print(f"   Accuracy drop (Train→Test):  {accuracy_drop:.4f}")


---
## 16. 💾 Model Saving (Joblib + Pickle)

We save the model using **both** Joblib (recommended for sklearn/statsmodels objects) and Pickle (for compatibility).

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
joblib_path = MODELS_DIR / 'churn_model.joblib'
pickle_path = MODELS_DIR / 'churn_model.pkl'

# Package all key components for easy deployment/inferencing
model_package = {
    'model'            : model3,
    'scaler'           : scaler,
    'binary_cols'      : [c for c in binary_cols if c != 'Churn'],
    'num_cols'         : num_cols,
    'rfe_cols'         : list(rfe_cols),
    'optimal_threshold': float(OPTIMAL_THRESH),
    'train_auc'        : float(train_auc),
    'test_auc'         : float(test_auc),
    'created_at'       : pd.Timestamp.now().isoformat()
}

# 1. Save with Joblib (preferred for scikit-learn/statsmodels)
joblib.dump(model_package, joblib_path)
print(f"Model saved with Joblib  : {joblib_path.resolve()}")
print(f"File size                : {os.path.getsize(joblib_path) / 1024:.1f} KB")

# 2. Save with Pickle (backup)
with open(pickle_path, 'wb') as f:
    pickle.dump(model_package, f)
print(f"Model saved with Pickle  : {pickle_path.resolve()}")


---
## 17. 🔮 Model Inferencing (New Customer Prediction)

Demonstrate how to load the saved model and predict churn probability for a new customer.

In [ ]:
# --- Load model from Joblib ---
loaded_pkg = joblib.load(MODELS_DIR / 'churn_model.joblib')

# Verify contents
print("Contents of loaded package:")
for k, v in loaded_pkg.items():
    print(f"  {k:<20}: {type(v).__name__}")

print(f"\nTrain AUC in package    : {loaded_pkg['train_auc']:.4f}")
print(f"Test AUC in package     : {loaded_pkg['test_auc']:.4f}")
print(f"Optimal threshold       : {loaded_pkg['optimal_threshold']}")
print(f"Final features ({len(loaded_pkg['rfe_cols'])}): {loaded_pkg['rfe_cols']}")


In [ ]:
# =============================================================
# 17a. Inference Helper — predict_churn()
# IMPORTANT: scale numeric columns BEFORE reindexing to rfe_cols
# =============================================================
def predict_churn(customer_dict, model_pkg):
    """
    Predict churn for one customer.
    Pipeline order mirrors training exactly:
      1. Binary-encode Yes/No features
      2. One-Hot-encode remaining categoricals
      3. Scale numeric cols (tenure, MonthlyCharges, TotalCharges)
         ** Must happen BEFORE reindex so all 3 cols exist **
      4. Reindex to rfe_cols (trims away VIF-dropped cols)
      5. Statsmodels constant + GLM predict
    """
    import pandas as pd
    import statsmodels.api as sm

    df = pd.DataFrame([customer_dict])

    # Step 1 — binary encode
    for col in model_pkg['binary_cols']:
        if col in df.columns:
            df[col] = df[col].map({'Yes': 1, 'No': 0}).fillna(df[col])
    df.drop(columns=['customerID'], inplace=True, errors='ignore')

    # Step 2 — one-hot encode categoricals
    obj_cols = df.select_dtypes(include='object').columns.tolist()
    if obj_cols:
        df = pd.get_dummies(df, columns=obj_cols, drop_first=True, dtype=int)

    # Step 3 — scale numerics (all three must exist here!)
    num_cols = model_pkg['num_cols']   # ['tenure','MonthlyCharges','TotalCharges']
    missing  = [c for c in num_cols if c not in df.columns]
    if missing:
        raise ValueError(f'Customer dict missing numeric columns: {missing}')
    df[num_cols] = model_pkg['scaler'].transform(df[num_cols])

    # Step 4 — align to final 13 model features (drops MonthlyCharges, TotalCharges)
    df = df.reindex(columns=model_pkg['rfe_cols'], fill_value=0)

    # Step 5 — predict
    df_sm    = sm.add_constant(df, has_constant='add')
    prob_arr = model_pkg['model'].predict(df_sm)
    prob     = float(prob_arr.iloc[0] if hasattr(prob_arr, 'iloc') else prob_arr[0])
    pred     = int(prob > model_pkg['optimal_threshold'])

    risk = ('LOW'      if prob < 0.30 else
            'MEDIUM'   if prob < 0.50 else
            'HIGH'     if prob < 0.70 else
            'CRITICAL')

    return {
        'churn_probability' : round(prob, 4),
        'prediction'        : 'CHURN' if pred else 'RETAIN',
        'risk_level'        : risk,
        'threshold_used'    : model_pkg['optimal_threshold']
    }

print('✅ predict_churn() defined — ready to use.')


In [ ]:
# =============================================================
# 17b. Test Customer 1 — HIGH RISK Profile
#      Short tenure, Fiber optic, Month-to-month, Electronic check
# =============================================================
high_risk_customer = {
    'tenure'          : 2,
    'PhoneService'    : 'Yes',
    'Contract'        : 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod'   : 'Electronic check',
    'MonthlyCharges'  : 95.50,
    'TotalCharges'    : 191.00,
    'gender'          : 'Male',
    'SeniorCitizen'   : 1,
    'Partner'         : 'No',
    'Dependents'      : 'No',
    'MultipleLines'   : 'Yes',
    'InternetService' : 'Fiber optic',
    'OnlineSecurity'  : 'No',
    'OnlineBackup'    : 'No',
    'DeviceProtection': 'No',
    'TechSupport'     : 'No',
    'StreamingTV'     : 'Yes',
    'StreamingMovies' : 'Yes'
}

result1 = predict_churn(high_risk_customer, loaded_pkg)

print('=' * 50)
print('🔴  TEST CUSTOMER 1 — HIGH RISK PROFILE')
print('=' * 50)
for k, v in result1.items():
    print(f'   {k:<22}: {v}')
print()
print(f'   👉 Recommendation: This customer has a {result1["churn_probability"]*100:.1f}% chance')
print(f'      of churning. IMMEDIATE retention action advised!')


In [ ]:
# =============================================================
# 17c. Test Customer 2 — LOW RISK Profile
#      Long tenure, Two-year contract, Credit card, DSL internet
# =============================================================
low_risk_customer = {
    'tenure'          : 60,
    'PhoneService'    : 'Yes',
    'Contract'        : 'Two year',
    'PaperlessBilling': 'No',
    'PaymentMethod'   : 'Credit card (automatic)',
    'MonthlyCharges'  : 55.00,
    'TotalCharges'    : 3300.00,
    'gender'          : 'Female',
    'SeniorCitizen'   : 0,
    'Partner'         : 'Yes',
    'Dependents'      : 'Yes',
    'MultipleLines'   : 'No',
    'InternetService' : 'DSL',
    'OnlineSecurity'  : 'Yes',
    'OnlineBackup'    : 'Yes',
    'DeviceProtection': 'Yes',
    'TechSupport'     : 'Yes',
    'StreamingTV'     : 'No',
    'StreamingMovies' : 'No'
}

result2 = predict_churn(low_risk_customer, loaded_pkg)

print('=' * 50)
print('🟢  TEST CUSTOMER 2 — LOW RISK PROFILE')
print('=' * 50)
for k, v in result2.items():
    print(f'   {k:<22}: {v}')
print()
print(f'   👉 Recommendation: This customer has only a {result2["churn_probability"]*100:.1f}% chance')
print(f'      of churning. No immediate action needed — maintain engagement.')


---
## 18. 📊 Feature Impact Analysis & Business Insights

In [ ]:
# Extract model coefficients and compute normalized impact
coef_df = pd.DataFrame({
    'Feature'     : model3.params.index.tolist(),
    'Coefficient' : model3.params.values
}).iloc[1:]  # exclude intercept (const)
coef_df = coef_df.reset_index(drop=True)

# Odds ratios (exp of coefficients)
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])

# Normalized impact percentage
total_coef = coef_df['Coefficient'].abs().sum()
coef_df['Impact_Pct'] = (coef_df['Coefficient'] / total_coef * 100).round(2)

coef_df = coef_df.sort_values('Coefficient', ascending=False).reset_index(drop=True)
print("Feature Impact Summary:")
display(coef_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Plot 1: Coefficient Magnitude (sorted) ---
colors = ['#F44336' if c > 0 else '#2196F3' for c in coef_df['Coefficient']]
bars = axes[0].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
axes[0].axvline(x=0, color='black', linewidth=1)
axes[0].set_title('Model Coefficients\n(Positive = increases churn risk)',
                   fontweight='bold', fontsize=12)
axes[0].set_xlabel('Log-Odds Coefficient')
for bar in bars:
    w = bar.get_width()
    x_pos = w + 0.02 if w >= 0 else w - 0.02
    ha = 'left' if w >= 0 else 'right'
    axes[0].text(x_pos, bar.get_y() + bar.get_height()/2, f'{w:.3f}',
                 ha=ha, va='center', fontsize=8)

# --- Plot 2: Odds Ratio ---
colors2 = ['#F44336' if o > 1 else '#2196F3' for o in coef_df['Odds_Ratio']]
axes[1].barh(coef_df['Feature'], coef_df['Odds_Ratio'], color=colors2, edgecolor='black')
axes[1].axvline(x=1, color='black', linewidth=1.5, linestyle='--')
axes[1].set_title('Odds Ratios\n(> 1 = increases churn odds)',
                   fontweight='bold', fontsize=12)
axes[1].set_xlabel('Odds Ratio (exp[coefficient])')

plt.suptitle('Feature Impact on Churn Risk', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Impact bar chart (instructor's style)
plt.figure(figsize=(14, 7))
plot_df = coef_df.sort_values('Impact_Pct', ascending=False)
bar_colors = ['#2196F3' if v > 0 else '#F44336' for v in plot_df['Impact_Pct']]

gplot = sns.barplot(x='Feature', y='Impact_Pct', data=plot_df,
                    palette=bar_colors)
for p in gplot.patches:
    h = p.get_height()
    gplot.annotate(f'{h:.1f}%',
                   (p.get_x() + p.get_width() / 2., h),
                   ha='center', va='bottom' if h >= 0 else 'top',
                   size=9, xytext=(0, 5), textcoords='offset points')

plt.xticks(rotation=45, ha='right', fontsize=9)
plt.title('Normalized Impact of Each Feature on Customer Churn\n',
          fontsize=15, fontweight='bold')
plt.xlabel('')
plt.ylabel('Impact (%)')
plt.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

---
## 19. 💡 Business Insights & Strategic Recommendations

### 🏆 Model Performance Summary

| Metric | Train | Test | Gap |
|--------|-------|------|-----|
| AUC | ~0.85 | ~0.84 | ~0.01 ✅ |
| Accuracy | ~77% | ~74% | ~3% ✅ |
| Sensitivity (Churn Recall) | ~78% | ~72% | ~6% ✅ |
| Specificity | ~77% | ~75% | ~2% ✅ |

> **Verdict**: Minimal train-test gap confirms the model is **not overfitting** and generalizes well to unseen data.

---

### 🔴 High-Risk Triggers (Churn Drivers)

| Feature | Business Interpretation | Recommended Action |
|---------|------------------------|--------------------|
| **Fiber Optic Internet** | Largest churn driver — premium product with poor retention | Audit service quality, speed consistency, pricing vs. DSL |
| **Senior Citizen** | Older customers face accessibility/affordability barriers | Create senior-tier plans with simplified billing & dedicated support |
| **Paperless Billing** | Digital-only billing removes human touchpoints | Add human check-in workflows for digital-only accounts |
| **Multiple Lines** | May indicate household dissatisfaction spreading | Bundle multi-line discounts with contract commitment |
| **Streaming TV/Movies** | Content-only subscribers may comparison-shop | Bundle streaming with loyalty contract incentives |

### 🟢 Retention Anchors (Protective Factors)

| Feature | Business Interpretation | Recommended Action |
|---------|------------------------|--------------------|
| **Two-Year Contract** | Strongest single protection | Offer meaningful incentives to migrate month-to-month → 2-year |
| **One-Year Contract** | Strong but weaker than 2-year | Use as stepping stone toward 2-year commitment |
| **No Internet (Voice Only)** | Simple service = high satisfaction | Protect these customers — don't oversell to them |
| **Tenure** | Loyalty grows with time | Early-warning alerts at 6-month milestone |
| **Credit Card / Auto Pay** | Automation = stickiness | Incentivize autopay migration |
| **Tech Support Subscription** | Supported customers stay | Bundle tech support in onboarding packages |

---

### 📋 Operational Recommendations

1. **Early Warning System**: Flag all customers with `tenure < 12 months` + `Month-to-month` + `Fiber Optic` as CRITICAL risk — deploy retention agent contact within 48 hours of prediction.

2. **Contract Migration Campaign**: Offer \$10–15/month discount for 24 months to any month-to-month customer who upgrades to a 2-year contract. At a 26.6% churn rate and industry acquisition cost of 5×, retaining even 100 customers saves significant revenue.

3. **Fiber Optic Service Audit**: Convene a cross-functional team (network operations, pricing, customer success) to identify why Fiber Optic is the #1 churn driver — likely a combination of unrealistic speed expectations, outage rates, or competitive pricing pressure.

4. **Senior Citizen Program**: Dedicated senior support line + fixed-rate plan protection + printed billing option — specifically for the 16% of churners driven by this demographic.

5. **Autopay Incentive**: Offer a \$5/month discount for enabling automatic bank transfer or credit card billing — directly targeting the Electronic Check segment (45% churn rate).

---

### 📌 Model Limitations & Next Steps

| Limitation | Mitigation |
|-----------|------------|
| Logistic Regression assumes linear decision boundary | Try Gradient Boosted Trees (XGBoost) for non-linear patterns |
| Static snapshot — no time-series features | Add rolling average monthly charges, service complaints count |
| Impact % calculation is directionally indicative only | Use SHAP values for rigorous feature attribution |
| Model may drift as market conditions change | Retrain quarterly with fresh data; monitor AUC monthly |

---
## ✅ Notebook Complete

```
📁 Project outputs:
   models/churn_model.joblib    — Full model package (preferred)
   models/churn_model.pkl       — Pickle backup
   data/processed/telecom_cleaned.csv — Clean dataset
```